# EDA: что за данные и как по ним валидироваться

Ноутбук отвечает на вопросы, от которых зависит дизайн решения:

1. Лежат ли объявления из `train` в боевом корпусе `benchmark_items`? От этого зависит, как строить локальную валидацию.
2. Насколько тексты запросов бенчмарка встречались в `train`? Это потолок для фич «запомнить по train».
3. Сколько релевантных объявлений у запроса, насколько полезны категория и локация.
4. Как выглядят тексты объявлений, то есть что подавать в лексический поиск.

Описание (`item_description_raw`) — основная часть объёма паркета. Здесь его не читаем, колонки берём выборочно.

In [1]:
import pandas as pd

D = "../dataset/"
Q = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]
tr = pd.read_parquet(
    D + "train.parquet",
    columns=Q
    + [
        "item_id",
        "item_title_raw",
        "item_infm_params_text",
        "item_category_id",
        "item_microcat_id",
        "item_location_id",
    ],
)
bq = pd.read_parquet(D + "benchmark_queries.parquet")
bi = pd.read_parquet(
    D + "benchmark_items.parquet",
    columns=[
        "item_id",
        "item_title_raw",
        "item_infm_params_text",
        "item_category_id",
        "item_microcat_id",
        "item_location_id",
    ],
)
print(f"train: {len(tr):,} пар, уникальных объявлений {tr.item_id.nunique():,}")
print(f"benchmark: {len(bq):,} запросов, корпус {len(bi):,} объявлений")

train: 497,673 пар, уникальных объявлений 344,825
benchmark: 2,452 запросов, корпус 189,212 объявлений


## 1. Объявления train ↔ боевой корпус

In [2]:
share = tr.item_id.drop_duplicates().isin(set(bi.item_id)).mean()
print(f"доля уникальных объявлений train, которые есть в benchmark_items: {share:.1%}")

доля уникальных объявлений train, которые есть в benchmark_items: 5.3%


**Вывод.** Пересечение около 5%. Для большинства запросов train правильного ответа в боевом корпусе нет, поэтому **валидировать по `benchmark_items` нельзя**.

Валидационный корпус собираем из объявлений train: все релевантные для отложенных запросов плюс случайные до 189 212, то есть размера боевого корпуса. Так сложность поиска совпадает с боевой (`src/eval.py::make_holdout`).

Следствие для будущих фич: лукап «запрос → конкретные `item_id` из train» почти бесполезен. Переносимы только обобщённые сигналы: категории, микрокатегории, слова.

## 2. Тексты запросов бенчмарка в train

In [3]:
def norm(s: pd.Series) -> pd.Series:
    return s.str.lower().str.strip()


seen = norm(bq.search_query).isin(set(norm(tr.search_query)))
print(f"уникальных текстов запросов в train: {tr.search_query.nunique():,}")
print(f"уникальных текстов в бенчмарке: {bq.search_query.nunique():,} (на {len(bq):,} query_id)")
print(f"доля запросов бенчмарка, чей текст встречался в train: {seen.mean():.1%}")

уникальных текстов запросов в train: 74,529
уникальных текстов в бенчмарке: 2,452 (на 2,452 query_id)
доля запросов бенчмарка, чей текст встречался в train: 37.0%


**Вывод.** В бенчмарке каждый `query_id` — отдельный текст, 37% текстов встречались в train. Для этой трети из train можно достать, в какие микрокатегории и к каким формулировкам пользователи уходили по этому запросу. Остальные 63% train напрямую не покрывает, и там работает только перенос через слова и эмбеддинги.

Для валидации отсюда правило: одна группа на уникальный текст, как в бенчмарке.

## 3. Размер релевантного множества

В train нет `query_id`, поэтому запрос — это уникальный кортеж признаков `search_*`.

In [4]:
g = tr.groupby(Q, dropna=False).item_id.nunique()
print(f"групп запросов: {len(g):,}")
print(
    (g.clip(upper=5).value_counts(normalize=True).sort_index().rename({5: "5+"}) * 100).round(1).astype(str)
    + "%"
)

групп запросов: 354,463
item_id
1     85.0%
2      9.5%
3      2.6%
4      1.1%
5+     1.8%
Name: proportion, dtype: str


**Вывод.** У 85% запросов ровно одно релевантное объявление. Это подтверждает «обычно одно-два» из условия. Recall@50 здесь почти бинарный: попали в 50 или нет.

## 4. Категория и локация

In [5]:
print("search_category:", tr.search_category.value_counts().to_dict())
print("search_category в бенчмарке:", bq.search_category.value_counts().to_dict())
print(
    f"item_category_id: {tr.item_category_id.nunique()} значений, "
    f"item_microcat_id: {tr.item_microcat_id.nunique()}"
)
print(
    f"доля пар, где локация объявления = локации поиска: "
    f"{(tr.search_location_id == tr.item_location_id).mean():.1%}"
)
print(
    f"поиск с доставкой в бенчмарке: {bq.search_is_delivery_search.mean():.1%}, "
    f"пустые фильтры: {(bq.search_infm_params_text.fillna('') == '').mean():.1%}"
)

search_category: {114: 497635, 0: 34, 81: 2, 10: 2}
search_category в бенчмарке: {114: 2230, 0: 222}
item_category_id: 8 значений, item_microcat_id: 212
доля пар, где локация объявления = локации поиска: 83.1%
поиск с доставкой в бенчмарке: 0.0%, пустые фильтры: 63.1%


**Выводы.**
- `search_category` в train почти константа (114, «Услуги»). В бенчмарке у 9% запросов стоит 0, в train такого почти нет. Учиться на этом поле не на чем, как признак его не используем. Полезнее `item_microcat_id`: 212 значений, их можно предсказывать по тексту запроса из train.
- **В 83% пар объявление из той же локации, что и поиск.** Это самый сильный нелексический сигнал. Использовать его как буст, а не как жёсткий фильтр: метрика — полнота, и каждое шестое попадание лежит в другой локации.
- Доставки в бенчмарке нет, фильтры пустые у 63% запросов.

## 5. Что подавать в лексический поиск

In [6]:
print("длина item_title_raw:")
print(bi.item_title_raw.str.len().describe()[["mean", "50%", "max"]].round(0))
print("длина item_infm_params_text:")
print(bi.item_infm_params_text.fillna("").str.len().describe()[["mean", "50%", "max"]].round(0))
print()
print(bi.item_infm_params_text.iloc[0][:400])

длина item_title_raw:
mean     34.0
50%      35.0
max     100.0
Name: item_title_raw, dtype: float64
длина item_infm_params_text:
mean      974.0
50%       738.0
max     19068.0
Name: item_infm_params_text, dtype: float64

Вид услуги Компьютерная помощь Место оказания услуг пр-т Ленина Тип стоимости за услугу Начальная цена График работы от 34200 График работы до 82800 Время работы, с 09:30 Время работы, до 23:00 Начальная цена Тип стоимости за услугу Стоимость 500 Услуга Диагностика компьютера Начальная цена Тип стоимости за услугу Стоимость 500 Услуга Начальная цена Тип стоимости за услугу Стоимость 500 Услуга Нач


**Вывод.** Заголовок короткий, ~34 символа. `item_infm_params_text` — полуструктурный блоб в среднем ~970 символов. В начале полезное «Вид услуги … Тип услуги …», дальше шаблонный шум: «График работы», «Тип стоимости», цены. Шум ложно матчится с запросами, например «работа в клининг» тянет объявления с «График работы».

Замер на валидации (символьный TF-IDF, Recall@50):

| текст объявления | Recall@50 |
|---|---|
| только заголовок | 0.272 |
| заголовок + параметры целиком | 0.278 |
| **заголовок + первые 200 символов параметров** | **0.301** |
| заголовок + первые 400 символов параметров | 0.289 |

В baseline идёт третий вариант (`src/baseline.py::item_text`).